# UzAniDub on Kaggle

Dubs videos into Uzbek with the [UzAniDub](https://github.com/alwaysneedhelp/UzAniDub) pipeline.

**Before running:**

1. **Settings → Accelerator → GPU T4 ×2** (speech synthesis is the slow stage; the T4 makes it several times faster than CPU).
2. **Settings → Internet → On** (needs a phone-verified Kaggle account). Code and models (~10 GB) download on every fresh session.
3. **Add Input → upload your videos as a private dataset.** Every `.mp4` / `.mkv` / `.mov` / `.webm` under `/kaggle/input` gets dubbed, unless you set `VIDEOS` below.
4. **Add-ons → Secrets → Add Secret** → label `GEMINI_API_KEY`, value your key from [aistudio.google.com/apikey](https://aistudio.google.com/apikey), and tick it so it's attached to this notebook. Needed while `TRANSLATOR = "gemini"`; set `TRANSLATOR = "madlad"` to translate locally with no key. Optional: `HF_TOKEN` for pyannote diarization.

Dubbed files land in `/kaggle/working/output`. Download them from the Output panel, or use **Save Version → Save & Run All** to keep them with the notebook.

In [ ]:
# ---- Settings ---------------------------------------------------------------
REPO_URL = "https://github.com/alwaysneedhelp/UzAniDub.git"
BRANCH = "main"

TRANSLATOR = "gemini"   # "gemini" (needs the GEMINI_API_KEY secret) or "madlad" (local, no key)

VIDEOS = []            # explicit paths; empty = every video under /kaggle/input
NAMES = ""             # e.g. "Gojo,Toji,Megumi" -- kept verbatim through translation
GLOSSARY = "assets/jjk_glossary.yaml"  # fixed renderings for show terms; repo-relative or a /kaggle/input path, "" for none
NUM_SPEAKERS = None    # diarizer hint, e.g. 3
EXTRA_ARGS = []        # any other dubtool flags, e.g. ["--no-clone-voices"]

In [ ]:
import os, subprocess, sys
print(sys.version)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "No GPU -- dubbing will run on CPU (slow)"
!ffmpeg -version | head -1
!df -h /kaggle/working /tmp

## 1. Code

In [ ]:
WORK = "/kaggle/working"  # only dubbed videos, logs and the voice bank live here (saved with a version)
REPO = "/tmp/UzAniDub"    # code + ~10 GB of models stay out of the saved output
COSY_COMMIT = "074ca6dc9e80a2f424f1f74b48bdd7d3fea531cc"  # CosyVoice revision the pipeline was verified on

if not os.path.isdir(REPO):
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}
os.chdir(REPO)

if not os.path.isdir("poc/CosyVoice/.git"):
    !git clone https://github.com/FunAudioLLM/CosyVoice.git poc/CosyVoice
    !git -C poc/CosyVoice checkout -q {COSY_COMMIT}
    !git -C poc/CosyVoice submodule update --init --recursive
!git log -1 --oneline

## 2. Python packages

Installs the project's pinned dependencies into Kaggle's environment, **except** torch/torchaudio/numpy: Kaggle's preinstalled CUDA build of torch is kept so it matches the GPU driver.

In [ ]:
import re, tomllib

deps = tomllib.load(open("pyproject.toml", "rb"))["project"]["dependencies"]
keep_kaggle = ("torch", "torchaudio", "numpy")
reqs = [d for d in deps if re.split(r"[=<>!~;@ \[]", d, maxsplit=1)[0].strip().lower() not in keep_kaggle]
reqs += ["modelscope", "huggingface_hub"]  # CosyVoice imports modelscope at load time
reqs += tomllib.load(open("pyproject.toml", "rb"))["project"]["optional-dependencies"]["gemini"]  # google-genai
# faster-whisper passes av.open(metadata_errors=...), which the PyAV preinstalled on Kaggle rejects
reqs += ["av==16.1.0"]
open("/tmp/kaggle_reqs.txt", "w").write("\n".join(reqs) + "\n")

!pip install -q --progress-bar off -r /tmp/kaggle_reqs.txt
!pip install -q --progress-bar off --no-deps -e .
!python -c "import torch, numpy; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| numpy', numpy.__version__)"

## 3. Models (CosyVoice2 + navoiy-tts Uzbek checkpoint)

In [ ]:
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"  # thousands of progress lines can freeze the browser tab
from huggingface_hub import hf_hub_download, snapshot_download

snapshot_download("FunAudioLLM/CosyVoice2-0.5B", local_dir="poc/models/CosyVoice2-0.5B")
for f in ("emotion_600h_joint.pt", "emotions_40h.json", "uztts/normalize.py", "LICENSE", "NOTICE"):
    hf_hub_download("aisha-org/navoiy-tts", f, local_dir="poc/models/navoiy-tts")
open("poc/models/navoiy-tts/uztts/__init__.py", "a").close()
!du -sh poc/models/*

## 4. Dub

In [ ]:
import glob, shlex

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for key in ("HF_TOKEN", "GEMINI_API_KEY"):
        try:
            os.environ[key] = secrets.get_secret(key)
            print("using secret", key)
        except Exception:
            pass
except ImportError:
    pass

if TRANSLATOR == "gemini" and not os.environ.get("GEMINI_API_KEY"):
    raise RuntimeError("TRANSLATOR is 'gemini' but no GEMINI_API_KEY secret is attached: add it under "
                       "Add-ons -> Secrets (and tick it for this notebook), or set TRANSLATOR = 'madlad'")

videos = VIDEOS or sorted(
    p for ext in ("mp4", "mkv", "mov", "webm")
    for p in glob.glob(f"/kaggle/input/**/*.{ext}", recursive=True)
)
print(f"{len(videos)} video(s):", *videos, sep="\n  ")

OUT = f"{WORK}/output"
LOGS = f"{OUT}/logs"
os.makedirs(LOGS, exist_ok=True)
failed = []
for src in videos:
    name = os.path.splitext(os.path.basename(src))[0]
    out, log_path = f"{OUT}/{name}_uz.mp4", f"{LOGS}/{name}.log"
    if os.path.exists(out):
        print("already dubbed:", out); continue
    cmd = ["dubtool", src, "-o", out, "--voice-bank-dir", f"{WORK}/voice_bank", "-v"]
    if TRANSLATOR == "gemini": cmd += ["--config", "configs/gemini.yaml"]
    if NAMES: cmd += ["--names", NAMES]
    if GLOSSARY: cmd += ["--glossary", GLOSSARY]
    if NUM_SPEAKERS: cmd += ["--num-speakers", str(NUM_SPEAKERS)]
    cmd += EXTRA_ARGS
    print("\n$", shlex.join(cmd), f"\n  full log: {log_path}", flush=True)
    # The full log (model downloads, per-line timing) goes to a file: streamed
    # into the cell it is thousands of lines and can freeze the browser tab.
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            if "Stage " in line or "ERROR" in line or "Traceback" in line:
                print(line.rstrip()[:200], flush=True)
    if proc.wait() != 0:
        failed.append(src)
        print(f"FAILED (exit {proc.returncode}) -- last lines of the log:")
        !tail -n 30 "{log_path}"
print("\nfailed:", failed or "none")

## 5. Results

Download the dubbed files from the **Output** panel (`/kaggle/working/output`). To watch one here, run `Video(path, embed=True)` from `IPython.display`, but an embedded video makes the editor sluggish, so delete that cell's output afterwards.

In [ ]:
for f in sorted(glob.glob(f"{OUT}/*_uz.mp4")):
    dur = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", f],
                         capture_output=True, text=True).stdout.strip()
    print(f"{f}  {os.path.getsize(f) / 1e6:.1f} MB  {float(dur or 0):.1f}s")